<h1>Section 1. Loading Data</h1>

Configure the XCP-D connectivity and DCAN motion-QC paths, then run the package workflow. Files are matched on subject, session, task, and run; the selected atlas is checked against matrix labels and order. Connectivity features include Schaefer 9/19-network summaries and left, right, and bilateral hippocampus/amygdala ROI-to-network measures.

### Motion metrics (FD=0.3 default)

| Metric | Meaning |
| --- | --- |
| `mean_fd_remaining` | Mean FD among retained frames |
| `frames_remaining`, `frames_total`, `frames_censored` | Retained, total, and censored volumes |
| `seconds_remaining`, `fraction_remaining` | Retained scan duration and proportion |
| `skip` | DCAN QC skip indicator |

Section 2 compares remaining scan duration at FD=0.3, 0.4, and 0.5. The 4S metadata assign cortical parcels to Schaefer networks and label subcortex/cerebellum explicitly.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "rsx_analyzer").is_dir()),
    Path.cwd(),
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from rsx_analyzer import (
    export_analysis_results_pdf,
    export_excel,
    plot_feature_motion_associations,
    plot_outlier_summary,
    plot_threshold_comparison,
    plot_triple_networks,
    robust_outlier_screen,
    run_workflow,
    summarize_multiple_runs,
    triple_network_summary,
)

CONN_MATS_PATH = Path("data/4S356/conn_mats")
MOTION_PATH = Path("data/dcan_qc")
ATLAS_TSV_PATH = PROJECT_ROOT / "atlases" / "atlas-4S356Parcels_dseg.tsv"
TASK_FILTER = "rest"
FD_THRESHOLD = 0.3
OUTPUT_DIRECTORY = CONN_MATS_PATH.parent.parent
OUTPUT_EXCEL_PATH = OUTPUT_DIRECTORY / "rs-X1_analysis.xlsx"
OUTPUT_PDF_PATH = OUTPUT_DIRECTORY / "rs-X1_results.pdf"

In [ ]:
analysis = run_workflow(
    CONN_MATS_PATH,
    MOTION_PATH,
    ATLAS_TSV_PATH,
    task_filter=TASK_FILTER,
    fd_threshold=FD_THRESHOLD,
)

atlas = analysis.atlas
conn_files = analysis.discovery.conn_files
motion_files = analysis.discovery.motion_files
connectivity_df = analysis.connectivity.connectivity_df
feature_columns = list(analysis.connectivity.feature_columns)
motion_df = analysis.motion.motion_df
motion_load_errors_df = analysis.motion.motion_load_errors
run_level_df = analysis.run_level.run_level_df
coverage_summary = analysis.run_level.coverage_summary
missingness_report = analysis.run_level.missingness_report
motion_threshold_df = analysis.motion_threshold_data
threshold_summary = analysis.threshold_table
master_df = analysis.master.master_df
df_merged_runs = analysis.merged_runs
qc_motion_association = analysis.motion_associations

print(f"Atlas: {ATLAS_TSV_PATH.name} ({len(atlas.names)} parcels)")
print(f"Loaded {len(connectivity_df)} matrices and {len(motion_df)} readable motion files.")
print(f"Computed {len(feature_columns)} network and ROI-to-network features.")
print(f"Matched readable runs: {len(master_df)}; subject/session rows after run averaging: {len(df_merged_runs)}.")

In [ ]:
print("Subject coverage by session and run:")
display(coverage_summary)
print("Unmatched or unreadable subject/session/task/run files:")
display(missingness_report)
if not motion_load_errors_df.empty:
    print("Unreadable HDF5 motion files:")
    display(motion_load_errors_df)

example_features = [
    "Cont_within_9net", "Cont_X_Default_9net", "ContA_within_19net",
    "ContA_X_DefaultA_19net", "LH_Hippocampus_to_Cont_9net",
    "RH_Hippocampus_to_Cont_9net", "Bilateral_Hippocampus_to_Cont_9net",
    "Bilateral_Amygdala_to_DefaultA_19net",
]
display(connectivity_df[["subject", "session", "task", "run", *example_features]].head())

<h1>Section 2. Quality Control</h1>

FD=0.3 is the default for run-level features. QC associations are exploratory and repeated runs/sessions are not independent observations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_threshold_comparison(motion_threshold_df, ax=ax)
ax.set_xlabel("Framewise displacement threshold (mm)")
ax.set_ylabel("Remaining scan duration (seconds)")
ax.set_title("Remaining duration across motion thresholds")
fig.tight_layout()
plt.show()
display(threshold_summary)

In [ ]:
print("Master run-level data:")
display(master_df.iloc[:5, :12])
print("Features most associated with remaining seconds at FD=0.3:")
display(qc_motion_association[[
    "feature", "n_runs", "spearman_rho", "spearman_q", "pearson_r", "pearson_p"
]].head(20))
fig, ax = plt.subplots(figsize=(9, 7))
plot_feature_motion_associations(qc_motion_association, top_n=15, ax=ax)
ax.set_xlabel("Spearman correlation with remaining seconds (FD=0.3)")
ax.set_title("Connectivity features most associated with motion-related data loss")
fig.tight_layout()
plt.show()

multiple_runs_by_session, multiple_run_subjects = summarize_multiple_runs(df_merged_runs)
print("Subjects with multiple matched runs by session:")
display(multiple_runs_by_session)
print(f"Subjects with multiple runs in at least one session: {multiple_run_subjects['SubjectID'].nunique()}")
display(multiple_run_subjects)

<h2>Feature Outliers</h2>

Robust MAD/IQR scores flag extreme subject/session averages at |robust z| > 3.5. These are review flags, not automatic exclusions.

In [ ]:
OUTLIER_ROBUST_Z_THRESHOLD = 3.5
outlier_results = robust_outlier_screen(
    df_merged_runs,
    feature_columns,
    z_threshold=OUTLIER_ROBUST_Z_THRESHOLD,
)
outlier_feature_summary = outlier_results["feature_summary"]
outlier_feature_flags = outlier_results["feature_flags"]
outlier_subject_report = outlier_results["subject_report"]
unscorable_features = outlier_results["unscorable"]
print(
    f"Screened {len(feature_columns)} features across {len(df_merged_runs)} subject/session rows; "
    f"flagged {len(outlier_subject_report)} subjects in "
    f"{(outlier_feature_summary['n_outlier_rows'] > 0).sum()} features."
)
print(f"Unscorable features due to zero robust scale: {len(unscorable_features)}")
display(outlier_subject_report)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_outlier_summary(outlier_feature_summary, outlier_subject_report, axes=axes)
fig.tight_layout()
plt.show()

<h2>Triple-Network Connectivity</h2>

Compare within- and between-system connectivity for Control (CEN), Default (DMN), and Salience (SN). The 19-network values use parcel-pair-weighted summaries.

In [ ]:
TRIPLE_NETWORKS = {
    "Control (CEN)": {"network9": ("Cont",), "network19": ("ContA", "ContB", "ContC")},
    "Default (DMN)": {"network9": ("Default",), "network19": ("DefaultA", "DefaultB", "DefaultC")},
    "Salience (SN)": {"network9": ("SalVentAttn",), "network19": ("SalVentAttnA", "SalVentAttnB")},
}
network19_counts = pd.Series(
    atlas.networks["network_label_19network"]
).value_counts().to_dict()
triple_network_df, triple_summary = triple_network_summary(
    df_merged_runs,
    network19_counts,
    TRIPLE_NETWORKS,
)
triple_feature_order = [f"{name} within" for name in TRIPLE_NETWORKS] + [
    f"{left} x {right}"
    for index, left in enumerate(TRIPLE_NETWORKS)
    for right in list(TRIPLE_NETWORKS)[index + 1:]
]
display(triple_summary)
triple_ax = plot_triple_networks(triple_network_df, order=triple_feature_order)
triple_figure = triple_ax.figure
triple_ax.set_title("Triple-network connectivity by atlas resolution")
triple_figure.tight_layout()
plt.show()

<h2>Export Results</h2>

Write the analysis tables and print-ready QC report beside the input dataset.

In [ ]:
export_excel(
    {
        "merged_runs": df_merged_runs,
        "QC": outlier_subject_report,
        "Trinetwork": triple_summary,
    },
    OUTPUT_EXCEL_PATH,
    figure=triple_figure,
    figure_sheet="Trinetwork",
    figure_title="Triple-network connectivity by atlas resolution",
    figure_subtitle="19-network summary uses parcel-pair-weighted means.",
)
export_analysis_results_pdf(
    OUTPUT_PDF_PATH,
    atlas_name=ATLAS_TSV_PATH.name,
    n_atlas_parcels=len(atlas.names),
    fd_threshold=FD_THRESHOLD,
    n_conn_files=len(conn_files),
    n_motion_files=len(motion_files),
    n_motion_readable=len(motion_df),
    n_master_rows=len(master_df),
    n_merged_run_rows=len(df_merged_runs),
    n_features=len(feature_columns),
    n_flagged_subjects=len(outlier_subject_report),
    threshold_summary=threshold_summary,
    motion_associations=qc_motion_association,
    outlier_feature_summary=outlier_feature_summary,
    outlier_subject_report=outlier_subject_report,
    triple_summary=triple_summary,
    triple_network_df=triple_network_df,
    triple_feature_order=triple_feature_order,
    outlier_z_threshold=OUTLIER_ROBUST_Z_THRESHOLD,
)
print(f"Excel workbook: {OUTPUT_EXCEL_PATH}")
print(f"Print-ready PDF: {OUTPUT_PDF_PATH}")